In [ ]:
# requirements.txt file create karo
with open('requirements.txt', 'w') as f:
    f.write("""torch>=2.0.0
transformers>=4.30.0
datasets>=2.12.0
peft>=0.5.0
accelerate>=0.20.0
evaluate>=0.4.0
gradio>=3.50.0
huggingface_hub>=0.16.0
sentencepiece>=0.1.99
protobuf>=3.20.0
scipy>=1.10.0
bitsandbytes>=0.41.0
""")

# Check karo file bani ya nahi
!cat requirements.txt

torch>=2.0.0
transformers>=4.30.0
datasets>=2.12.0
peft>=0.5.0
accelerate>=0.20.0
evaluate>=0.4.0
gradio>=3.50.0
huggingface_hub>=0.16.0
sentencepiece>=0.1.99
protobuf>=3.20.0
scipy>=1.10.0
bitsandbytes>=0.41.0


In [ ]:
!pip install -r requirements.txt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.2 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
from datasets import load_dataset

# 'dialogsum' dataset - bilkul samsum jaisa (dialogue + summary)
dataset = load_dataset("knkarthick/dialogsum")

# R&D ke liye sirf 500 samples le rahe hain (taaki training jaldi ho)
train_data = dataset["train"].select(range(500))
test_data = dataset["test"].select(range(100))

print(f"✅ Training samples: {len(train_data)}")
print(f"✅ Test samples: {len(test_data)}")

# Ek example dekhte hain
print("\n--- Example ---")
print("Dialogue:", train_data[0]["dialogue"])
print("Summary:", train_data[0]["summary"])

README.md:   0%|          | 0.00/4.65k [00:00<?, ?B/s]

train.csv: reconstructing file:   0%|          |  0.00B / 11.3MB            

train.csv: downloading bytes:           |  0.00B            

validation.csv:   0%|          | 0.00/442k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

✅ Training samples: 500
✅ Test samples: 100

--- Example ---
Dialogue: #Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?
#Person2#: I found it would be a good idea to get a check-up.
#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.
#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?
#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.
#Person2#: Ok.
#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?
#Person2#: Yes.
#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.
#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.
#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.
#Person2#: Ok, thanks docto

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

# Tokenizer load karo (text ko numbers mein convert karega)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Base model load karo (250M parameters wala)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("✅ Tokenizer and Model loaded successfully!")
print(f"Model parameters: {model.num_parameters():,}")  # ~250M dikhega

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ Tokenizer and Model loaded successfully!
Model parameters: 247,577,856


In [ ]:
# Corrected preprocessing function - text_target use karo
def preprocess_function(examples):
    # Input (Dialogue) ko tokenize karo
    inputs = tokenizer(
        examples["dialogue"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

    # Target (Summary) ko tokenize karo - text_target parameter use karo
    # Yeh automatically decoder ki taraf tokenize karega
    labels = tokenizer(
        text_target=examples["summary"],   # <--- Yeha dekho! as_target_tokenizer ki jagah
        truncation=True,
        padding="max_length",
        max_length=64
    )

    inputs["labels"] = labels["input_ids"]
    return inputs

# Ab dataset pe apply karo
tokenized_train = train_data.map(preprocess_function, batched=True, remove_columns=train_data.column_names)
tokenized_test = test_data.map(preprocess_function, batched=True, remove_columns=test_data.column_names)

print(f"✅ Tokenization complete!")
print(f"Training samples: {len(tokenized_train)}")
print(f"Test samples: {len(tokenized_test)}")
print("\nSample tokenized input shape:", tokenized_train[0]["input_ids"][:10])

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

✅ Tokenization complete!
Training samples: 500
Test samples: 100

Sample tokenized input shape: [1713, 345, 13515, 536, 4663, 10, 2018, 6, 1363, 5]


In [ ]:
!pip install peft==0.15.1


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.0/411.0 kB 27.4 MB/s eta 0:00:00
  Attempting uninstall: peft
    Found existing installation: peft 0.20.0
    Uninstalling peft-0.20.0:
      Successfully uninstalled peft-0.20.0


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# LoRA Config
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

# Model pe LoRA apply karo
model = get_peft_model(model, lora_config)

# Dekho kitne parameters train honge
model.print_trainable_parameters()

trainable params: 884,736 || all params: 248,462,592 || trainable%: 0.3561


In [ ]:
!pip show transformers | grep Version


Version: 5.16.1


In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from transformers import DataCollatorForSeq2Seq

# Data Collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./my_summarizer_model",
    eval_strategy="steps",
    eval_steps=100,
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    num_train_epochs=3,
    predict_with_generate=True,
    generation_max_length=64,
    logging_steps=10,
    save_steps=500,
    report_to="none",
    fp16=True,
    push_to_hub=False,
)

# ✅ Trainer initialize karo - tokenizer hata diya, sirf data_collator rakha
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,   # Tokenizer yahan internally handle ho raha hai
)

print("✅ Trainer ready! Ab training shuru karne ke liye agla cell run karo.")

✅ Trainer ready! Ab training shuru karne ke liye agla cell run karo.


In [ ]:
trainer.train()

Step,Training Loss,Validation Loss
100,0.000000,nan
189,0.000000,nan


TrainOutput(global_step=189, training_loss=0.0, metrics={'train_runtime': 85.4536, 'train_samples_per_second': 17.553, 'train_steps_per_second': 2.212, 'total_flos': 515606446080000.0, 'train_loss': 0.0, 'epoch': 3.0})

In [ ]:
# 1. Dekho asli summary kya hai
print("Original Summary:", train_data[0]["summary"])

# 2. Dekho tokenized labels kya hain (numbers mein)
print("Labels IDs:", tokenized_train[0]["labels"])

# 3. In numbers ko wapas words mein badlo (decode karo)
print("Decoded Labels:", tokenizer.decode(tokenized_train[0]["labels"], skip_special_tokens=True))

Original Summary: Mr. Smith's getting a check-up, and Doctor Hawkins advises him to have one every year. Hawkins'll give some information about their classes and medications to help Mr. Smith quit smoking.
Labels IDs: [1363, 5, 3931, 31, 7, 652, 3, 9, 691, 18, 413, 6, 11, 7582, 12833, 77, 7, 7786, 7, 376, 12, 43, 80, 334, 215, 5, 12833, 77, 7, 31, 195, 428, 128, 251, 81, 70, 2287, 11, 11208, 12, 199, 1363, 5, 3931, 10399, 10257, 5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Decoded Labels: Mr. Smith's getting a check-up, and Doctor Hawkins advises him to have one every year. Hawkins'll give some information about their classes and medications to help Mr. Smith quit smoking.


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer
from transformers import DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, TaskType

print("1️⃣ Loading dataset...")
dataset = load_dataset("knkarthick/dialogsum")
train_data = dataset["train"].select(range(500))
test_data = dataset["test"].select(range(100))

print("2️⃣ Loading model & tokenizer...")
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("3️⃣ Tokenizing (with fixed labels)...")
def preprocess_function(examples):
    inputs = tokenizer(examples["dialogue"], truncation=True, padding="max_length", max_length=256)
    labels = tokenizer(text_target=examples["summary"], truncation=True, padding=False, max_length=64)
    inputs["labels"] = labels["input_ids"]
    return inputs

tokenized_train = train_data.map(preprocess_function, batched=True, remove_columns=train_data.column_names)
tokenized_test = test_data.map(preprocess_function, batched=True, remove_columns=test_data.column_names)

print("4️⃣ Applying LoRA...")
lora_config = LoraConfig(r=8, lora_alpha=32, target_modules=["q", "v"], lora_dropout=0.05, bias="none", task_type=TaskType.SEQ_2_SEQ_LM)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("5️⃣ Setting up Trainer...")
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
training_args = Seq2SeqTrainingArguments(
    output_dir="./my_summarizer_model",
    eval_strategy="steps",
    eval_steps=100,
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    num_train_epochs=3,
    predict_with_generate=True,
    generation_max_length=64,
    logging_steps=10,
    save_steps=500,
    report_to="none",
    fp16=True,
    push_to_hub=False,
)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
)
print("✅ Sab setup ho gaya! Ab training start karo.")

1️⃣ Loading dataset...
2️⃣ Loading model & tokenizer...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


3️⃣ Tokenizing (with fixed labels)...


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

4️⃣ Applying LoRA...
trainable params: 884,736 || all params: 248,462,592 || trainable%: 0.3561
5️⃣ Setting up Trainer...
✅ Sab setup ho gaya! Ab training start karo.


In [ ]:
trainer.train()

Step,Training Loss,Validation Loss
100,0.000000,nan
189,0.000000,nan


TrainOutput(global_step=189, training_loss=0.0, metrics={'train_runtime': 66.0437, 'train_samples_per_second': 22.712, 'train_steps_per_second': 2.862, 'total_flos': 515606446080000.0, 'train_loss': 0.0, 'epoch': 3.0})

In [ ]:
print("🔍 Checking what is inside tokenized_train right now...")

# 1. Pehle sample ki labels check karo
print("\n1. First sample labels (first 20 tokens):", tokenized_train[0]["labels"][:20])

# 2. Kya yeh saare -100 hain?
labels_check = tokenized_train[0]["labels"]
if all(t == -100 for t in labels_check[:10]):
    print("❌ ERROR: Labels sab -100 hain! (Ignore index)")
else:
    print("✅ Labels mein valid numbers hain (e.g., 1363, 5, ...)")

# 3. Batch collator se ek batch banakar dekhte hain (yeh sabse accurate hai)
from torch.utils.data import DataLoader

test_loader = DataLoader(tokenized_train, batch_size=4, collate_fn=data_collator)
batch = next(iter(test_loader))

print("\n2. Batch labels shape:", batch['labels'].shape)
print("3. Batch labels (pehli 3 rows, pehle 10 columns):")
print(batch['labels'][:3, :10])

# 4. Check karo ki kya koi -100 hai ya sab 0 hain?
if (batch['labels'] == -100).all():
    print("❌ FATAL: Saare labels -100 hain! Collator sabko ignore kar raha hai.")
elif (batch['labels'] == 0).all():
    print("❌ FATAL: Saare labels 0 hain! Sirf padding tokens hain.")
else:
    print("✅ Batch labels sahi dikh rahe hain!")

🔍 Checking what is inside tokenized_train right now...

1. First sample labels (first 20 tokens): [1363, 5, 3931, 31, 7, 652, 3, 9, 691, 18, 413, 6, 11, 7582, 12833, 77, 7, 7786, 7, 376]
✅ Labels mein valid numbers hain (e.g., 1363, 5, ...)

2. Batch labels shape: torch.Size([4, 48])
3. Batch labels (pehli 3 rows, pehle 10 columns):
tensor([[ 1363,     5,  3931,    31,     7,   652,     3,     9,   691,    18],
        [ 8667, 13156,  1217, 11066,    63,    21,   112, 12956,     7,     5],
        [ 1713,   345, 13515,   536,  4663,    31,     7,   479,    21,     3]])
✅ Batch labels sahi dikh rahe hain!


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer
from transformers import DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, TaskType

print("="*50)
print("🚀 LOADING DATA")
print("="*50)
dataset = load_dataset("knkarthick/dialogsum")
train_data = dataset["train"].select(range(500))
test_data = dataset["test"].select(range(100))

print("🤖 LOADING MODEL")
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("✂️ TOKENIZATION (labels padding=False)")
def preprocess_function(examples):
    inputs = tokenizer(examples["dialogue"], truncation=True, padding="max_length", max_length=256)
    # Labels ko padding=False rakho, collator manage karega
    labels = tokenizer(text_target=examples["summary"], truncation=True, padding=False, max_length=64)
    inputs["labels"] = labels["input_ids"]
    return inputs

tokenized_train = train_data.map(preprocess_function, batched=True, remove_columns=train_data.column_names)
tokenized_test = test_data.map(preprocess_function, batched=True, remove_columns=test_data.column_names)

print("🧠 LORA APPLY")
lora_config = LoraConfig(r=8, lora_alpha=32, target_modules=["q", "v"], lora_dropout=0.05, bias="none", task_type=TaskType.SEQ_2_SEQ_LM)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("⚙️ TRAINER SETUP (FP16 = FALSE, label_pad_token_id = -100)")

# 🔥🔥🔥 ULTIMATE FIX:
# 1. Collator ko explicitly batao ki pad token -100 hai (taaki loss mein ignore ho)
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100  # <--- Yeh force karo!
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./test_run",
    eval_strategy="steps",
    eval_steps=50,
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    num_train_epochs=1,               # 🔥 Sirf 1 epoch (5 min mein test ho jayega)
    predict_with_generate=True,
    generation_max_length=64,
    logging_steps=10,
    save_steps=100,
    report_to="none",
    fp16=False,                       # 🔥🔥🔥 FP16 HATA DIYA (is se glitch ho raha tha)
    push_to_hub=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
)

print("✅ Sab set! Ab training start hoti hai (1 epoch, 5-7 min)...")

# 🔥 Training Run
trainer.train()

print("\n🎉 Training complete! Loss kya aaya? Upar table mein dekho!")

🚀 LOADING DATA
🤖 LOADING MODEL


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


✂️ TOKENIZATION (labels padding=False)


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

🧠 LORA APPLY
trainable params: 884,736 || all params: 248,462,592 || trainable%: 0.3561
⚙️ TRAINER SETUP (FP16 = FALSE, label_pad_token_id = -100)
✅ Sab set! Ab training start hoti hai (1 epoch, 5-7 min)...


Step,Training Loss,Validation Loss
50,1.632731,1.534203
63,1.575829,1.527932



🎉 Training complete! Loss kya aaya? Upar table mein dekho!


In [ ]:
# 🚀 Training continue karte hain - ab total 3 epochs karenge
print("📈 Pehle 1 epoch ho chuka hai. Ab total 3 epochs ke liye set kar raha hoon...")

# Trainer ko batana hai ki total 3 epochs chalane hain
trainer.args.num_train_epochs = 3

# 🔥 Ab wapas train karo - 2 aur epochs (approx 2 minute lagega)
print("⏳ 2 aur epochs chal rahe hain (approx 2 min)...")
trainer.train()

print("\n🎉 Ab 3 epochs complete! Loss values check karo:")
print("🚀 Umeed hai ki loss ab 1.0 ke aas-paas ho gaya hoga!")

📈 Pehle 1 epoch ho chuka hai. Ab total 3 epochs ke liye set kar raha hoon...
⏳ 2 aur epochs chal rahe hain (approx 2 min)...


Step,Training Loss,Validation Loss
50,1.484141,1.442403
100,1.421673,1.427197
150,1.365960,1.415517
189,1.399336,1.419984



🎉 Ab 3 epochs complete! Loss values check karo:
🚀 Umeed hai ki loss ab 1.0 ke aas-paas ho gaya hoga!


In [ ]:
# 🚀 Ab total 5 epochs kar dete hain (2 aur)
print("📈 Loss ab 1.40 pe hai. 2 aur epochs se dekhte hain kya hota hai...")
print("⏳ Agar loss 1.3 tak gir gaya toh bahut badhiya, warna yahin rok denge.")

# Total epochs 5 kar do
trainer.args.num_train_epochs = 5

# 🔥 Training continue karo (2 aur epochs, ~1.5 minute)
trainer.train()

print("\n🎯 5 Epochs complete! Final loss values dekho:")
print("✅ Agar loss 1.3 ke aas-paas hai, toh model ready hai deploy ke liye!")

📈 Loss ab 1.40 pe hai. 2 aur epochs se dekhte hain kya hota hai...
⏳ Agar loss 1.3 tak gir gaya toh bahut badhiya, warna yahin rok denge.


Step,Training Loss,Validation Loss
50,1.422940,1.404836
100,1.380502,1.400906
150,1.286427,1.401213
200,1.369548,1.397217
250,1.439713,1.393509
300,1.330560,1.398655
315,1.500935,1.398761



🎯 5 Epochs complete! Final loss values dekho:
✅ Agar loss 1.3 ke aas-paas hai, toh model ready hai deploy ke liye!


In [ ]:
# 📦 Model Save aur Upload
from huggingface_hub import HfApi, create_repo

# 1. Pehle local save karo
print("💾 Saving model locally...")
model.save_pretrained("./my_summarizer_final")
tokenizer.save_pretrained("./my_summarizer_final")
print("✅ Local save complete!")

# 2. Hugging Face Hub pe upload karo (tumhara username 'prabhat-30' hai)
print("☁️ Uploading to Hugging Face Hub...")
model.push_to_hub("prabhat-30/flan-t5-dialogsum-summarizer")
tokenizer.push_to_hub("prabhat-30/flan-t5-dialogsum-summarizer")
print("✅ Model uploaded successfully!")
print("🔗 Model link: https://huggingface.co/prabhat-30/flan-t5-dialogsum-summarizer")

💾 Saving model locally...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Local save complete!
☁️ Uploading to Hugging Face Hub...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-6aad35ff-0d0a68563f4dec4127455506;0bbb96c7-6e42-4ad4-9070-2e823577b5e7)

Repository Not Found for url: https://huggingface.co/api/models/prabhat-30/flan-t5-dialogsum-summarizer/preupload/main.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.
Note: Creating a commit assumes that the repo already exists on the Huggingface Hub. Please use `create_repo` if it's not the case.

In [ ]:
# ============ CELL 1: Install (pinned versions, no build issues) ============
!pip install -q "transformers>=4.38,<4.45" peft==0.15.1 accelerate gradio sentencepiece safetensors huggingface_hub tokenizers==0.19.1

# ============ CELL 2: Gradio App (Fixed for Gradio 6.0+) ============
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftConfig, get_peft_model
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
import gradio as gr

BASE_MODEL   = "google/flan-t5-base"
ADAPTER_REPO = "prabhat-30/flan-t5-dialogsum-summarizer"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("🔌 Device:", device)

# --- Load base model + tokenizer ---
tokenizer  = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL).to(device)

# --- Read LoRA config ---
peft_config = PeftConfig.from_pretrained(ADAPTER_REPO)

# --- Wrap base model ---
model = get_peft_model(base_model, peft_config)

# --- Manually load adapter weights ---
try:
    adapter_path = hf_hub_download(ADAPTER_REPO, "adapter_model.safetensors")
    adapter_weights = load_file(adapter_path, device=device)
except Exception:
    adapter_path = hf_hub_download(ADAPTER_REPO, "adapter_model.bin")
    adapter_weights = torch.load(adapter_path, map_location=device)

missing, unexpected = model.load_state_dict(adapter_weights, strict=False)
print(f"✅ Loaded adapter | missing: {len(missing)} | unexpected: {len(unexpected)}")

# --- Merge LoRA into base → faster inference ---
model = model.merge_and_unload()
model.to(device).eval()

# ============ Inference function (fixed task = "summarize") ============
def summarize_dialogue(dialogue):
    input_text = f"summarize: {dialogue.strip()}"
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4,
            early_stopping=True
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

# ============ CLEAN UI (No extra options, no flagging, default theme) ============
demo = gr.Interface(
    fn=summarize_dialogue,
    inputs=gr.Textbox(
        lines=10,
        label="💬 Enter Dialogue / Conversation",
        placeholder="Person1: Hi, how are you?\nPerson2: I'm good! Just got a new job.\nPerson1: Congratulations!"
    ),
    outputs=gr.Textbox(
        label="📝 Summary",
        lines=5
    ),
    title="📌 Smart Dialogue Summarizer",
    description=(
        "This AI tool generates **concise summaries** of conversational dialogues. "
        "Built by fine-tuning **Google FLAN-T5-base** on the **DialogSum** dataset, "
        "it captures key points from multi-turn conversations — ideal for chat logs, "
        "customer support transcripts, and meeting notes."
    ),
    submit_btn="✨ Summarize"
)

print("🚀 App launching... Link neeche aayega!")
demo.launch(share=True, debug=False)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.0/321.0 kB 27.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 127.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.3/32.3 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.5 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable whee

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


adapter_config.json:   0%|          | 0.00/767 [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 3.56MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

✅ Loaded adapter | missing: 428 | unexpected: 144
🚀 App launching... Link neeche aayega!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3b8365b48f9be1d054.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# ====================================================
# 📁 PROJECT FILES: app.py + requirements.txt
# Ye files download karo aur Hugging Face Spaces pe upload karo
# ====================================================

# ---------- app.py (Main Gradio App) ----------
app_code = """
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftConfig, get_peft_model
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
import gradio as gr

# ---------- Config ----------
BASE_MODEL = "google/flan-t5-base"
ADAPTER_REPO = "prabhat-30/flan-t5-dialogsum-summarizer"
device = "cuda" if torch.cuda.is_available() else "cpu"

# ---------- Load Model ----------
def load_model():
    print("🔌 Loading tokenizer and base model...")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL).to(device)

    print("📐 Loading LoRA config...")
    peft_config = PeftConfig.from_pretrained(ADAPTER_REPO)
    model = get_peft_model(base_model, peft_config)

    print("📥 Loading adapter weights manually...")
    try:
        adapter_path = hf_hub_download(ADAPTER_REPO, "adapter_model.safetensors")
        adapter_weights = load_file(adapter_path, device=device)
    except Exception:
        adapter_path = hf_hub_download(ADAPTER_REPO, "adapter_model.bin")
        adapter_weights = torch.load(adapter_path, map_location=device)

    missing, unexpected = model.load_state_dict(adapter_weights, strict=False)
    print(f"✅ Loaded | missing: {len(missing)} | unexpected: {len(unexpected)}")

    print("🔗 Merging LoRA into base model...")
    model = model.merge_and_unload()
    return tokenizer, model.to(device).eval()

tokenizer, model = load_model()
print("✅ Model ready for inference!")

# ---------- Inference ----------
def summarize_dialogue(dialogue):
    if not dialogue.strip():
        return "Please enter a dialogue."

    input_text = f"summarize: {dialogue.strip()}"
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4,
            early_stopping=True
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# ---------- Gradio UI ----------
demo = gr.Interface(
    fn=summarize_dialogue,
    inputs=gr.Textbox(
        lines=10,
        label="💬 Enter Dialogue / Conversation",
        placeholder="Person1: Hi, how are you?\\nPerson2: I'm good! Just got a new job.\\nPerson1: Congratulations!"
    ),
    outputs=gr.Textbox(
        label="📝 Summary",
        lines=5
    ),
    title="📌 Smart Dialogue Summarizer",
    description=(
        "This AI tool generates **concise summaries** of conversational dialogues. "
        "Built by fine-tuning **Google FLAN-T5-base** on the **DialogSum** dataset, "
        "it captures key points from multi-turn conversations — ideal for chat logs, "
        "customer support transcripts, and meeting notes."
    ),
    submit_btn="✨ Summarize"
)

if __name__ == "__main__":
    demo.launch(share=False)  # Space pe share=False rakho, port automatically handle hota hai
"""

# ---------- requirements.txt ----------
requirements = """
torch
transformers>=4.38,<4.45
peft==0.15.1
accelerate
gradio
sentencepiece
safetensors
huggingface_hub
tokenizers==0.19.1
"""

# ---------- Files Write Karo ----------
with open("app.py", "w") as f:
    f.write(app_code)

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("✅ Files created successfully in Colab!")

# ---------- Download Karne ke liye Button ----------
from google.colab import files
files.download("app.py")
files.download("requirements.txt")

print("📂 Files download ho gayi hongi tumhare laptop pe. Check karo!")

✅ Files created successfully in Colab!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📂 Files download ho gayi hongi tumhare laptop pe. Check karo!


In [ ]:
# Step 1: Pehle Python version check karo (sirf jaanne ke liye)
import sys
print(sys.version)

3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


In [ ]:
# Step 2: Rust compiler install karo
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable

info: downloading installer
warn: it looks like you have an existing rustup settings file at:
warn: /root/.rustup/settings.toml
info: profile set to default
info: default host tuple is x86_64-unknown-linux-gnu
warn: Updating existing toolchain, profile choice will be ignored
info: syncing channel updates for stable-x86_64-unknown-linux-gnu
info: default toolchain set to stable-x86_64-unknown-linux-gnu

  stable-x86_64-unknown-linux-gnu unchanged - rustc 1.98.1 (48a229cea 2026-09-01)


Rust is installed now. Great!

To get started you may need to restart your current shell.
This would reload your PATH environment variable to include
Cargo's bin directory ($HOME/.cargo/bin).

To configure your current shell, you need to source the
corresponding env file under $HOME/.cargo.

Consider running the right command for your shell (note the leading DOT):
. "$HOME/.cargo/env" # For sh/ash/dash/pdksh/bash
cargo:rerun-if-env-changed=CC_x86_64-unknown-linux-gnu
CC_x86_64-unknown-linux-gnu = None
car

In [ ]:
# Step 3: Rust ka path environment mein add karo (IMPORTANT: ye same cell ya turant baad wali cell mein hona chahiye)
import os
os.environ["PATH"] = f"/root/.cargo/bin:{os.environ['PATH']}"

# Verify Rust install hua
!rustc --version

rustc 1.98.1 (48a229cea 2026-09-01)


In [ ]:
!pip install -q -U transformers datasets peft accelerate sentencepiece indic-transliteration evaluate rouge_score sacrebleu

In [ ]:
# ============ CELL 1 (REVISED): Clean pinned install ============
!pip install -q -U pip
!pip install -q "transformers==4.44.2" "datasets==2.19.1" "peft==0.15.1" "accelerate==0.33.0" sentencepiece indic-transliteration evaluate rouge_score sacrebleu

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> tokenizers


In [ ]:
from datasets import load_dataset, concatenate_datasets, Dataset
import re

dialogsum = load_dataset("knkarthick/dialogsum")
print(dialogsum)

def clean_labels(text):
    # Person1:/Person2: labels hata do taaki model bina labels ke bhi samjhe
    return re.sub(r'#?Person\d#?:?\s*', '', text).strip()

def make_summarize_examples(split, remove_labels=False):
    src = [ (clean_labels(d) if remove_labels else d) for d in split["dialogue"] ]
    tgt = split["summary"]
    inputs  = [f"summarize: {d}" for d in src]
    return Dataset.from_dict({"input_text": inputs, "target_text": tgt})

train_normal    = make_summarize_examples(dialogsum["train"], remove_labels=False)
train_augmented = make_summarize_examples(dialogsum["train"], remove_labels=True)

summarization_train = concatenate_datasets([train_normal, train_augmented]).shuffle(seed=42)
print("Summarization train size:", len(summarization_train))
print(summarization_train[0])

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})
Summarization train size: 24920
{'input_text': "summarize: #Person1#: Hello, Mr. Summerfield. How are you today?\n#Person2#: Very well. Thank you, Ms. Green.\n#Person1#: What can I do for you?\n#Person2#: Well, unfortunately, there is a problem with the order we received from you yesterday. It seems we haven't seen the right quantity of manuals to support the telephone system.\n#Person1#: Oh, dear, that's bad news. I'm very sorry to hear that, and you don't know how many packs are without manuals?\n#Person2#: No, because we haven't opened every pack. But in several of those that have been opened there are none, no manuals.\n#Person1#: I'm very 

In [ ]:
import transformers, datasets, peft
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)

transformers: 5.16.1
datasets: 4.8.5
peft: 0.15.1


In [ ]:
from datasets import load_dataset, Dataset
import random
random.seed(42)
N_PER_LANG = 8000

def sample_dataset(ds, n):
    n = min(n, len(ds))
    idx = random.sample(range(len(ds)), n)
    return ds.select(idx)

# Hindi - ye already sahi chal raha tha
hi_ds = load_dataset("cfilt/iitb-english-hindi", split="train")
print("Hindi dataset columns:", hi_ds.column_names)
hi_ds = sample_dataset(hi_ds, N_PER_LANG)

# German - FIX: "opus100" -> "Helsinki-NLP/opus-100"
de_ds = load_dataset("Helsinki-NLP/opus-100", "de-en", split="train")
print("\nGerman dataset columns:", de_ds.column_names)
print("Sample:", de_ds[0])
de_ds = sample_dataset(de_ds, N_PER_LANG)

# French - FIX: same repo
fr_ds = load_dataset("Helsinki-NLP/opus-100", "en-fr", split="train")
print("\nFrench dataset columns:", fr_ds.column_names)
print("Sample:", fr_ds[0])
fr_ds = sample_dataset(fr_ds, N_PER_LANG)

# Hinglish
hing_ds = load_dataset("findnitai/english-to-hinglish", split="train")
print("\nHinglish dataset columns:", hing_ds.column_names)
print("Sample:", hing_ds[0])
hing_ds = sample_dataset(hing_ds, N_PER_LANG)

README.md:   0%|          | 0.00/3.14k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  190MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 85.7kB            

...

README.md:   0%|          | 0.00/367 [00:00<?, ?B/s]

hinglish_upload_v1.json: reconstructing file:   0%|          |  0.00B / 27.1MB            

hinglish_upload_v1.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/189102 [00:00<?, ? examples/s]


Hinglish dataset columns: ['translation']
Sample: {'translation': {'en': "What's the name of the movie", 'hi_ng': 'film ka kya naam hai', 'source': 1}}


In [ ]:
from datasets import concatenate_datasets, Dataset

# Pehle Hinglish dataset ka structure check karo (columns already print ho chuka hoga upar)
print(hing_ds.column_names)
print(hing_ds[0])

['translation']
{'translation': {'en': 'Set a reminder for my wedding party tomorrow at 8 pm .', 'hi_ng': 'meri wedding party ke liye kal sham 8 bajhe ka ek reminder set karen .', 'source': 0}}


In [ ]:
def normalize_hi(example):
    t = example["translation"]
    return {
        "input_text": "translate English to Hindi: " + t["en"],
        "target_text": t["hi"]
    }

def normalize_de(example):
    t = example["translation"]
    return {
        "input_text": "translate English to German: " + t["en"],
        "target_text": t["de"]
    }

def normalize_fr(example):
    t = example["translation"]
    return {
        "input_text": "translate English to French: " + t["en"],
        "target_text": t["fr"]
    }

def normalize_hinglish(example):
    # Case 1: nested under 'translation'
    if "translation" in example:
        t = example["translation"]
        en_key = "en" if "en" in t else list(t.keys())[0]
        hing_key = [k for k in t.keys() if k != en_key][0]
        return {
            "input_text": "translate English to Hinglish: " + t[en_key],
            "target_text": t[hing_key]
        }
    # Case 2: flat columns
    else:
        en_col = "en" if "en" in example else "source"
        hing_col = "hi_ng" if "hi_ng" in example else "target"
        return {
            "input_text": "translate English to Hinglish: " + example[en_col],
            "target_text": example[hing_col]
        }

# Normalize sab datasets
hi_norm = hi_ds.map(normalize_hi, remove_columns=hi_ds.column_names)
de_norm = de_ds.map(normalize_de, remove_columns=de_ds.column_names)
fr_norm = fr_ds.map(normalize_fr, remove_columns=fr_ds.column_names)
hing_norm = hing_ds.map(normalize_hinglish, remove_columns=hing_ds.column_names)

# Sanity check — sab ka format same hona chahiye
for name, ds in [("Hindi", hi_norm), ("German", de_norm), ("French", fr_norm), ("Hinglish", hing_norm)]:
    print(f"\n{name} sample:", ds[0])

# Sab ko ek saath concat karo
translation_ds = concatenate_datasets([hi_norm, de_norm, fr_norm, hing_norm])
translation_ds = translation_ds.shuffle(seed=42)

print("\nTotal combined examples:", len(translation_ds))
print("Final sample:", translation_ds[0])

# Train/validation split
split_ds = translation_ds.train_test_split(test_size=0.05, seed=42)
train_ds = split_ds["train"]
val_ds = split_ds["test"]
print("Train size:", len(train_ds), "| Val size:", len(val_ds))

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]


Hindi sample: {'input_text': "translate English to Hindi: When the banyan 's leaves are still pink and tender, the delicate Map butterfly comes to feast on the sweet sticky smear of syrup on them.", 'target_text': 'जब बरगद के नए पत्ते गुलाबी और नर्म होते हैं, तब नाजुक सी मैप तितली उसके मधुर-चिपचिपे रस को चूसने आती है। '}

German sample: {'input_text': 'translate English to German: Take someone else.', 'target_text': 'einfach jemand anderen.'}

French sample: {'input_text': 'translate English to French: 114 He will say: You did tarry but a little-- had you but known (it):', 'target_text': "114 Il dira: Vous n'y avez demeuré que peu [de temps], si seulement vous saviez."}

Hinglish sample: {'input_text': 'translate English to Hinglish: Set a reminder for my wedding party tomorrow at 8 pm .', 'target_text': 'meri wedding party ke liye kal sham 8 bajhe ka ek reminder set karen .'}

Total combined examples: 32000
Final sample: {'input_text': 'translate English to German: This was not meant

In [ ]:
from datasets import load_dataset

# Poora DialogSum load karo (na ki sirf 500 examples wala subset)
dialogsum = load_dataset("knkarthick/dialogsum")
print(dialogsum)
print(dialogsum["train"][0])

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})
{'id': 'train_0', 'dialogue': "#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?\n#Person2#: I found it would be a good idea to get a check-up.\n#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.\n#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?\n#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.\n#Person2#: Ok.\n#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?\n#Person2#: Yes.\n#Person1#: Smoking is

In [ ]:
def normalize_summary(example):
    return {
        "input_text": "summarize: " + example["dialogue"],
        "target_text": example["summary"]
    }

dialogsum_train = dialogsum["train"].map(normalize_summary, remove_columns=dialogsum["train"].column_names)
dialogsum_val = dialogsum["validation"].map(normalize_summary, remove_columns=dialogsum["validation"].column_names)

print("DialogSum train size:", len(dialogsum_train))
print("Sample:", dialogsum_train[0])

Map:   0%|          | 0/12460 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

DialogSum train size: 12460
Sample: {'input_text': "summarize: #Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?\n#Person2#: I found it would be a good idea to get a check-up.\n#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.\n#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?\n#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.\n#Person2#: Ok.\n#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?\n#Person2#: Yes.\n#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.\n#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.\n#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.\n#Person2#: Ok, thanks do

In [ ]:
import re
import random
random.seed(42)

def strip_speaker_tags(dialogue):
    # #Person1#: aur #Person2#: tags hata do, sirf lines rakho
    text = re.sub(r"#Person\d+#:\s*", "", dialogue)
    return text.strip()

In [ ]:
def augment_summary_example(example, drop_prob=0.3):
    dialogue = example["dialogue"]
    if random.random() < drop_prob:
        dialogue = strip_speaker_tags(dialogue)
    return {"input_text": "summarize: " + dialogue, "target_text": example["summary"]}

dialogsum_train = dialogsum["train"].map(
    lambda ex: augment_summary_example(ex, drop_prob=0.3),
    remove_columns=dialogsum["train"].column_names
)
# dialogsum_val already normal hai (jaisa hai waisa rehne do — validation pe hum real-world format hi test karte hain)

print("DialogSum train size:", len(dialogsum_train))
print("Sample 1:", dialogsum_train[0])
print("Sample 2:", dialogsum_train[1])

Map:   0%|          | 0/12460 [00:00<?, ? examples/s]

DialogSum train size: 12460
Sample 1: {'input_text': "summarize: #Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?\n#Person2#: I found it would be a good idea to get a check-up.\n#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.\n#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?\n#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.\n#Person2#: Ok.\n#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?\n#Person2#: Yes.\n#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.\n#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.\n#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.\n#Person2#: Ok, thanks 

In [ ]:
from datasets import concatenate_datasets

combined_train = concatenate_datasets([dialogsum_train, train_ds])
combined_train = combined_train.shuffle(seed=42)

combined_val = concatenate_datasets([dialogsum_val, val_ds])

print("Total combined train size:", len(combined_train))
print("Total combined val size:", len(combined_val))
print("Sample:", combined_train[0])
print("Sample:", combined_train[1])

Total combined train size: 42860
Total combined val size: 2100
Sample: {'input_text': 'translate English to Hinglish: How long will it take to get to Maine tonight', 'target_text': 'Aaj raat Maine tak jane ke liye kitni der lagegi'}
Sample: {'input_text': 'translate English to Hindi: luxuriance', 'target_text': 'विलासिता'}


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_INPUT_LEN = 256
MAX_TARGET_LEN = 128

def tokenize_fn(examples):
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding="max_length"
    )
    labels = tokenizer(
        text_target=examples["target_text"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding="max_length"
    )
    labels["input_ids"] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = combined_train.map(tokenize_fn, batched=True, remove_columns=["input_text", "target_text"])
tokenized_val = combined_val.map(tokenize_fn, batched=True, remove_columns=["input_text", "target_text"])

print("Tokenized train:", len(tokenized_train))
print("Tokenized val:", len(tokenized_val))

Map:   0%|          | 0/42860 [00:00<?, ? examples/s]

Map:   0%|          | 0/2100 [00:00<?, ? examples/s]

Tokenized train: 42860
Tokenized val: 2100


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = "/content/drive/MyDrive/flan-t5-multitask-checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Checkpoints yaha save honge:", CHECKPOINT_DIR)

Mounted at /content/drive
Checkpoints yaha save honge: /content/drive/MyDrive/flan-t5-multitask-checkpoints


In [ ]:
from transformers import AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model, TaskType

base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q", "v"],  # T5 attention ke query/value projections
    bias="none"
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


trainable params: 1,769,472 || all params: 249,347,328 || trainable%: 0.7096


In [ ]:
from transformers.optimization import Adafactor, AdafactorSchedule

training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    logging_steps=50,          # zyada frequent logging, jaldi pata chalega NaN aaya to
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    predict_with_generate=True,
    fp16=False,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    max_grad_norm=1.0,
)

# Fresh model reload zaroori hai (agar abhi purana corrupted object use kar rahe ho)
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model = get_peft_model(base_model, lora_config)

optimizer = Adafactor(
    model.parameters(),
    scale_parameter=True,
    relative_step=False,
    warmup_init=False,
    lr=1e-4,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    optimizers=(optimizer, None),   # (optimizer, lr_scheduler) — None means default scheduler
)

trainer.train()

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Step,Training Loss,Validation Loss
500,4.695566,2.061821
1000,4.831444,2.058628
1500,4.582371,2.054509
2000,4.621783,2.049568
2500,4.651742,2.044336
3000,4.790968,2.038912
3500,4.512912,2.033643
4000,4.418750,2.028623
4500,4.624391,2.024132
5000,4.666456,2.020184


Step,Training Loss,Validation Loss
500,4.695566,2.061821
1000,4.831444,2.058628
1500,4.582371,2.054509
2000,4.621783,2.049568
2500,4.651742,2.044336
3000,4.790968,2.038912
3500,4.512912,2.033643
4000,4.418750,2.028623
4500,4.624391,2.024132
5000,4.666456,2.020184


In [ ]:
import os
# MyDrive ke andar jitne bhi folders hain, unki list aayegi
print(os.listdir("/content/drive/MyDrive"))

['Classroom', 'thin_film_interference.gslides', 'Int-hackathon team details (1).docx', '1280 Tech bit[1].docx', 'Image to PDF 20250929 12.05.52.pdf', '3. Pandas Manipulation — Cleaning & Feature Creat....gsheet', 'in excel sheet.gsheet', 'my ews pdf.pdf', 'Screenshot 2026-01-10 170759.png', 'Excel Data Export and Analysis.gsheet', 'डेटाबेस टेबल पैरामीटर्स भरना.gsheet', 'CSV Data Generation and Next Steps.gsheet', 'prabhat _ certificate .pdf', 'NPTEL_GERMAN_ASSIGNMENT_MARKS.pdf', 'major_sem_4_haal_ticket.pdf', 'WhatsApp Image 2026-05-04 at 12.34.43 AM.jpeg', 'Colab Notebooks', 'my_transfered_files', 'mujhe yeha samaj hi aya hai palha : yeha mod kya....gdoc', 'now for this.gdoc', '1759348717665.jpg', 'nptel_certificate.jpg', '1777025647607.jpg', 'google_cloud.pdf', 'leetcode_badges.pdf', 'Sih_prototype_video', 'Teammates Information.gsheet', 'flan-t5-multitask-checkpoints']


In [ ]:
import os
checkpoints = os.listdir("/content/drive/MyDrive/flan-t5-multitask-checkpoints")
checkpoints = sorted(checkpoints, key=lambda x: int(x.split("-")[-1]) if x.split("-")[-1].isdigit() else 0)
print(checkpoints)

['checkpoint-6500', 'checkpoint-7000', 'checkpoint-7500']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Version pin/force install NAHI karna — jo bhi pehle se installed hai (jisse training hui thi) wahi use karo
import transformers, peft
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)

Mounted at /content/drive
transformers: 5.16.1
peft: 0.20.0


In [ ]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftConfig, get_peft_model
from peft.utils.save_and_load import set_peft_model_state_dict   # ye safe hai, is_torchao check trigger nahi karta
from safetensors.torch import load_file
import torch, os

MODEL_NAME = "google/flan-t5-base"
LATEST_CHECKPOINT = "/content/drive/MyDrive/flan-t5-multitask-checkpoints/checkpoint-7500"

tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

peft_config = PeftConfig.from_pretrained(LATEST_CHECKPOINT)
model = get_peft_model(base_model, peft_config)

safetensors_path = os.path.join(LATEST_CHECKPOINT, "adapter_model.safetensors")
bin_path = os.path.join(LATEST_CHECKPOINT, "adapter_model.bin")

if os.path.exists(safetensors_path):
    adapter_weights = load_file(safetensors_path)
else:
    adapter_weights = torch.load(bin_path, map_location="cpu")

# Proper PEFT-aware loading — naming/prefix ka mismatch khud handle karta hai
load_result = set_peft_model_state_dict(model, adapter_weights)
print("Load result:", load_result)

# --- Sanity check: adapter apply hua ya nahi, generate karke dekho ---
test_input = "summarize: #Person1#: Hi, are you free tomorrow? #Person2#: Yes, what's up? #Person1#: Let's catch up for coffee. #Person2#: Sounds great!"
inputs = tokenizer(test_input, return_tensors="pt")
output = model.generate(**inputs, max_new_tokens=60)
print("Test summary output:", tokenizer.decode(output[0], skip_special_tokens=True))

model = model.merge_and_unload()

SAVE_DIR = "/content/drive/MyDrive/flan-t5-multitask-final-v2"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("✅ Final model saved to Drive:", SAVE_DIR)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Load result: _IncompatibleKeys(missing_keys=['base_model.model.shared.weight', 'base_model.model.encoder.embed_tokens.weight', 'base_model.model.encoder.block.0.layer.0.SelfAttention.q.base_layer.weight', 'base_model.model.encoder.block.0.layer.0.SelfAttention.k.weight', 'base_model.model.encoder.block.0.layer.0.SelfAttention.v.base_layer.weight', 'base_model.model.encoder.block.0.layer.0.SelfAttention.o.weight', 'base_model.model.encoder.block.0.layer.0.SelfAttention.relative_attention_bias.weight', 'base_model.model.encoder.block.0.layer.0.layer_norm.weight', 'base_model.model.encoder.block.0.layer.1.DenseReluDense.wi_0.weight', 'base_model.model.encoder.block.0.layer.1.DenseReluDense.wi_1.weight', 'base_model.model.encoder.block.0.layer.1.DenseReluDense.wo.weight', 'base_model.model.encoder.block.0.layer.1.layer_norm.weight', 'base_model.model.encoder.block.1.layer.0.SelfAttention.q.base_layer.weight', 'base_model.model.encoder.block.1.layer.0.SelfAttention.k.weight', 'base_model.mo

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Final model saved to Drive: /content/drive/MyDrive/flan-t5-multitask-final-v2


In [ ]:
from huggingface_hub import HfApi, create_repo

REPO_ID  = "prabhat-30/flan-t5-dialogsum-summarizer"
SAVE_DIR = "/content/drive/MyDrive/flan-t5-multitask-final-v2"

# Pehle confirm karo local files sahi hain
import os
print(os.listdir(SAVE_DIR))

['config.json', 'generation_config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json']


In [ ]:
from huggingface_hub import login
login()

In [ ]:
api = HfApi()
create_repo(REPO_ID, exist_ok=True)

api.upload_folder(
    folder_path=SAVE_DIR,
    repo_id=REPO_ID,
    repo_type="model",
    commit_message="Clean re-upload: full multitask model with config.json"
)

print("✅ Re-uploaded cleanly:", f"https://huggingface.co/{REPO_ID}")

✅ Re-uploaded cleanly: https://huggingface.co/prabhat-30/flan-t5-dialogsum-summarizer


In [ ]:
from transformers import AutoConfig
config = AutoConfig.from_pretrained(REPO_ID)
print(config.model_type)   # "t5" print hona chahiye

config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

t5


In [ ]:
# Yehi model jo already Colab mein load hai (SAVE_DIR se), usी se directly test karo
test_input = "translate English to Hindi: I am going to the market"
inputs = tokenizer(test_input, return_tensors="pt")

output_ids = model.generate(
    **inputs,
    max_new_tokens=128,
    num_beams=4,
    early_stopping=True,
)

print("Raw output ids:", output_ids)
print("Decoded (skip_special_tokens=True):", tokenizer.decode(output_ids[0], skip_special_tokens=True))
print("Decoded (skip_special_tokens=False):", tokenizer.decode(output_ids[0], skip_special_tokens=False))

Raw output ids: tensor([[0, 3, 2, 3, 2, 1]])
Decoded (skip_special_tokens=True):  
Decoded (skip_special_tokens=False): <pad> <unk> <unk></s>


In [ ]:
sample_hindi = "मैं बाजार जा रहा हूं"
token_ids = tokenizer(sample_hindi).input_ids
tokens = tokenizer.convert_ids_to_tokens(token_ids)
print("Token IDs:", token_ids)
print("Tokens:", tokens)

Token IDs: [3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 1]
Tokens: ['▁', '<unk>', '▁', '<unk>', '▁', '<unk>', '▁', '<unk>', '▁', '<unk>', '</s>']


In [ ]:
# Summarization Test
sample_dialogue = ''' Person1: Did you finish the project?
Person2: Yes, I submitted it yesterday.
Person1: Great work! '''
inputs = tokenizer(f"summarize: {sample_dialogue}", return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=50)
print("Summary:", tokenizer.decode(outputs[0], skip_special_tokens=True))

# Hinglish Test
inputs = tokenizer("translate English to Hinglish: Hello, how are you?", return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=50)
print("Hinglish:", tokenizer.decode(outputs[0], skip_special_tokens=True))

Summary: Person2 has submitted the project.
Hinglish: Hello, how are you?


In [ ]:
# 🔥 Multi-Task Testing Function
def test_model(task_prefix, input_text):
    prompt = f"{task_prefix}: {input_text}"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = model.generate(**inputs, max_new_tokens=50, num_beams=4)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result

# Test Cases
print("1️⃣ Summarization:")
print(test_model("summarize", "Person1: Did you finish the assignment? Person2: Yes, I submitted it today morning."))

print("\n2️⃣ English to German:")
print(test_model("translate English to German", "The weather is very nice today."))

print("\n3️⃣ English to French:")
print(test_model("translate English to French", "I love learning new languages."))

print("\n4️⃣ English to Hinglish:")
print(test_model("translate English to Hinglish", "Where are you going?"))

1️⃣ Summarization:
Person2 has submitted the assignment.

2️⃣ English to German:
The weather is very nice today.

3️⃣ English to French:
J'aime l'apprentissage de nouvelles langues.

4️⃣ English to Hinglish:
Where are you going?
